# Advanced Python
To master Python, you need to understand how things work under the hood. This phase explores Python's memory model, how objects are created and stored, object identity, scope rules, and advanced metaprogramming concepts.

## 1. Mutable vs. Immutable Objects
In Python, everything is an object, and every object falls into one of two categories based on whether its value can be modified after creation:

**Immutable Objects:** Once created, their state cannot be changed. If you modify an immutable object, Python actually creates a new object in memory.

Examples: int, float, bool, str, tuple, frozenset.

**Mutable Objects:** Their state and internal contents can be modified in place without changing their memory address (identity).

Examples: list, dict, set, bytearrays.

In [ ]:
# Immutable (Integer)
x = 10
print(id(x))
x += 1  # Creates a brand new integer object (11) and points x to it
print(id(x))  # Different ID

# Mutable (List)
my_list = [1, 2, 3]
print(id(my_list))
my_list.append(4)  # Modifies the list in place
print(id(my_list))  # Same ID

## 2. Shallow Copy vs. Deep Copy & The copy Module
When copying collections (like lists of lists), assignment (=) only copies references, not the actual data. To duplicate data, you use the copy module.

**Shallow Copy (copy.copy()):** Creates a new container object, but populates it with references to the child objects found in the original. If the inner objects are mutable, changes to them will affect both copies.

**Deep Copy (copy.deepcopy()):** Recursively creates a brand new container and brand new copies of all nested objects inside it. Nothing is shared with the original.

In [ ]:
import copy

original = [[1, 2], [3, 4]]

# Shallow copy
shallow = copy.copy(original)
shallow[0][0] = 99
print(original)  # Output: [[99, 2], [3, 4]] -> Inner list was modified!

# Deep copy
original_deep = [[1, 2], [3, 4]]
deep = copy.deepcopy(original_deep)
deep[0][0] = 99
print(original_deep)  # Output: [[1, 2], [3, 4]] -> Unaffected!

## 3. Memory Management, Reference Counting, and Garbage Collection
Python handles memory management automatically so you don't have to manually allocate or free bytes.

**Reference Counting:** Every Python object has a reference count tracking how many variables, containers, or functions point to it. When the reference count drops to zero, the object's memory is deallocated immediately. You can check this using sys.getrefcount().

**Garbage Collection (gc module):** While reference counting handles most objects, it fails to handle reference cycles (e.g., Object A points to Object B, and Object B points to Object A). Python includes a cyclic garbage collector (gc) that runs periodically to detect and clean up these isolated circular references.

## 4. String and Integer Interning
Interning is an optimization technique where Python stores only one copy of certain immutable objects in memory to save space and speed up comparisons.

**Strings:** Short strings, variable names, and strings that look like valid identifiers are automatically interned. You can manually intern arbitrary strings using sys.intern().

**Integers:** Python automatically interns small integers ranging from -5 to 256.

In [ ]:
a = 256
b = 256
print(a is b)  # True (interned)

x = 1000
y = 1000
print(x is y)  # False (not guaranteed to be interned)

## 5. Hashability and Object Identity (id(), is)
**Hashability:** An object is hashable if it has a __hash__() method and its hash value never changes throughout its lifetime. Immutable objects are generally hashable (and can be used as dictionary keys or set elements), while mutable objects are unhashable.

**Identity (id() and is):**

id(obj) returns the object's unique memory address (an integer).

The is operator checks if two variables point to the exact same object in memory (id(a) == id(b)), whereas == checks if their values are equal.

In [ ]:
a = [1, 2, 3]
b = [1, 2, 3]

print(a == b)  # True (values match)
print(a is b)  # False (different objects in memory)

## 6. Namespaces and Closures
**Namespaces:** A namespace is a mapping from names to objects (implemented as Python dictionaries). Python uses the LEGB rule to resolve names: Local, Enclosing (nested functions), Global (module level), and Built-in.

**Closures:** A closure is a nested function that remembers and has access to variables in its enclosing scope, even after the outer function has finished executing.

In [ ]:
def outer_func(msg):
    # Enclosing scope variable
    def inner_func():
        print(f"Message: {msg}")
    return inner_func

my_closure = outer_func("Hello, World!")
my_closure()  # Output: Message: Hello, World! (outer_func has finished, but msg is remembered)

## 7. Monkey Patching
Monkey patching refers to dynamically modifying or extending a class or module at runtime without altering its original source code. While powerful, it can make debugging confusing if overused.

In [ ]:
class Dog:
    def bark(self):
        return "Woof!"

# Original behavior
d = Dog()
print(d.bark())  # Output: Woof!

# Monkey patching a new method at runtime
def speak(self):
    return "Meow? (I'm an identity crisis dog)"

Dog.bark = speak

print(d.bark())  # Output: Meow? (I'm an identity crisis dog)

## 8. Metaclasses (High-Level Overview)
In Python, everything is an object, including classes. Just as an ordinary object is an instance of a class, a class itself is an instance of a metaclass.

The default metaclass in Python is type.

When you write a class statement, Python invokes the metaclass to create that class object.

Metaclasses are typically used in advanced libraries and frameworks (like Django ORM) to automatically validate, modify, or register classes when they are defined.

In [ ]:
# Creating a class dynamically using 'type' (the default metaclass)
# type(name, bases, dict)
MyDynamicClass = type('MyDynamicClass', (), {'x': 10, 'hello': lambda self: "Hi!"})

obj = MyDynamicClass()
print(obj.x)        # Output: 10
print(obj.hello())  # Output: Hi!

### Best Practices & Common Pitfalls
**Avoid Mutable Default Arguments:** Never use a mutable object (like a list or dict) as a default argument in a function definition. Because default arguments are evaluated once when the function is defined (not when it's called), modifications persist across function calls. Use None as a sentinel value instead.

**Use is None for Singletons:** Always check for None using is (if x is None:) rather than == because None is a singleton in Python, and is is faster and safer.

**Don't Overuse Monkey Patching:** While it's a great tool for hot-fixing libraries or writing tests (mocking), excessive monkey patching obscures code clarity and breaks static analysis tools.